实现各类分布式训练，测试模型主要是两类（首先本地vGPU测试单卡调试没问题而后去多卡进行测试）：1、视觉模型直接使用Resnet50然后在CIFAR10数据集上进行训练；2、使用Qwen0.5B模型进行文本分类训练

**代码要求**：1、尽可能的封装，方便直接用来进行复现代码

**文档要求**：1、在每一个分布式训练中都需要写好参数、参考地址、安装配置等尽可能详细就好；2、由于原理相通，所以**尽量多的去介绍调用框架的参数，对于具体原理不需要过多进行介绍**

# 基于原生 Pytorch 进行分布式训练

## DDP
> https://docs.pytorch.org/docs/2.11/notes/ddp.html


- [ ] 了解DDP底层：模型分配、如何执行 all-reduce操作
- [ ] 总结学习：https://docs.pytorch.org/docs/2.11/distributed.html#torch.distributed.all_reduce 其中所有带 distributed 的页面

`DDP`训练过程中原理：为不同设备都去复制一遍模型而后在不同的设备上都去输入一批数据在分别计算得到梯度之后进行 `all-reduce` 操作处理即可得到更新后模型在进行下一步优化。因此对于 `DDP`就需要关注几个点：1、模型分配；2、不同设备之间通信；3、数据处理。
### DDP 训练简易模板
```python
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler

class ExampleModel(nn.Module):
    ....

class ExampleDataset(Dataset):
    ....

def setup():
    """初始化分布式环境"""
    dist.init_process_group(backend="nccl") # 指定后端通信
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)

def cleanup():
    """销毁进程组"""
    dist.destroy_process_group()

def train():
    setup()
    local_rank = int(os.environ["LOCAL_RANK"])
    global_rank = int(os.environ["RANK"])

    dataset = ExampleDataset()
    # 使用 DistributedSampler 分解数据，防止不同 GPU 训练重复数据
    sampler = DistributedSampler(dataset, shuffle=True)
    dataloader = DataLoader(...,sampler=sampler,)

    model = ExampleModel().to(local_rank)
    # 处理模型
    model = DDP(model, device_ids=[local_rank])
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model.train()
    for epoch in range(5):
        sampler.set_epoch(epoch)
        for batch_idx, (data, target) in enumerate(dataloader):
            data, target = data.to(local_rank), target.to(local_rank)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            if batch_idx % 10 == 0 and global_rank == 0:
                print(f"Epoch {epoch} | Batch {batch_idx} | Loss: {loss.item():.4f}")
    if global_rank == 0:
        torch.save(model.module.state_dict(), "model_ddp.pth")
        print("模型已保存。")
    cleanup()

if __name__ == "__main__":
    train()
```
后续代码执行直接 `torchrun --nproc_per_node=4 train_script.py`，上述命令会启动 4 个 Python 进程，并自动为它们设置不同的环境变量：

进程 0: 设置 RANK=0, LOCAL_RANK=0, WORLD_SIZE=4

进程 1: 设置 RANK=1, LOCAL_RANK=1, WORLD_SIZE=4

进程 2: 设置 RANK=2, LOCAL_RANK=2, WORLD_SIZE=4

进程 3: 设置 RANK=3, LOCAL_RANK=3, WORLD_SIZE=4

那么在代码里直接使用 `os.environ.get("RANK")` 能直接拿到每一个显卡的“值”。**值得注意的是**：rank是所有的设备的序号，而 local_rank 是本机的序号，比如说有2台机器每一台都是4卡那么rank序号就是从0->7而 local_rank 则是从0->3而后另外一台机器也是 0->3
###  DDP 底层原理

## FSDP

- [ ] torch.compile 支持
- [ ] 更加多测试，如 dit 模型测试
- [ ] 保存训练参数配置，写到 tensorboard 文件夹中

# 基于 accelerate 进行分布式训练
> https://huggingface.co/docs/accelerate/index

最简单使用可以直接使用 `from transformers import trainer` 内部逻辑直接基于 accelerate 进行处理，支持 `deepepeed`、`fsdp`等训练方式
## 简单使用
**安装过程**直接通过：`pip install accelerate` 即可，使用过程直接通过如下命令：

```python
from accelerate import Accelerator
accelerator = Accelerator() # 初始化：指定日志记录（tensorboard等）、混合精度训练等

device = accelerator.device
model, optimizer, training_dataloader, scheduler = accelerator.prepare(
    model, optimizer, training_dataloader, scheduler
) # 通过 prepare 函数将模型、优化器、数据加载器、学习率调度器进行加速

for batch in training_dataloader:
    optimizer.zero_grad()
    inputs, targets = batch
    outputs = model(inputs)
    loss = loss_function(outputs, targets)
    # 区别普通的训练过程 accelerator 直接通过 backward 函数进行反向传播
    accelerator.backward(loss)
    optimizer.step()
    scheduler.step()
```

定义完代码之后启动脚本直接通过（对于启动支持的命令 `accelerate launch -h` 即可查看所有支持命令）：`accelerate launch main.py {--arg1} {--arg2}` **值得注意的是**，启动脚本中可以直接通过 `accelerate config` 而后会在终端中进行配置最后得到一个 `yaml` 的 config 文件，然后启动过程中直接：`accelerate launch --config_file {path/to/config/my_config_file.yaml} main.py {--arg1} {--arg2}` 即可，亦或者不想去使用 `yaml` 文件配置，可以直接在代码中指定好配置参数（比如直接使用 dataclass 去封装好训练参数），而后直接 `CUDA_VISIBLE_DEVICES=0,5,6,7 accelerate launch train_sft.py`（通过`CUDA_VISIBLE_DEVICES` 指定使用哪些 GPU进行训练）

## 初始化参数配置
根据[官方](https://huggingface.co/docs/accelerate/package_reference/accelerator)中参数配置主要使用参数如下（因为对于多种分布式训练如：DeepSpeed、FSDP等都在初始化配置中进行配置即可，因此下面代码中不去介绍这些参数，只介绍最常用的参数，在**后续具体使用过程中再去着重介绍**）：
```python
from accelerate import Accelerator

accelerator = Accelerator(
    mixed_precision= config.mixed_precision, # 混合精度训练: str=fp16/bf16/fp8/no
    gradient_accumulation_steps= config.gradient_accumulation_steps, # 梯度累积：int
    log_with= config.report_to, # 日志记录 list/str = ['wandb','tensorboard']等
    project_config= accelerator_project_config, # 项目配置 所有的文件保存路径
    kwargs_handlers= kwargs_handlers, # 配置参数处理
    )
```

## 分布式训练配置
### DDP
在`accelerate`中默认直接使用DDP训练因此不需要太多参数配置
### DeepSpeed

### FSDP2


# 基于 DeepSpeed 进行分布式训练

# 基于 Megatron-LM 进行分布式训练
> [https://github.com/nvidia/megatron-lm](https://github.com/nvidia/megatron-lm)

# 使用 Ray 分布式框架
https://docs.pytorch.org/tutorials/beginner/distributed_training_with_ray_tutorial.html